# Table 2 inputs — CDC ROI gradient CSVs

Exports mean signed derivatives within CDC weight-class windows (minmax mapping) for manuscript Table 2.

In [ ]:
# --- Configuration ---
import os
from pathlib import Path

import numpy as np
import pandas as pd

CEPH_ROOT = Path(os.environ.get("BMI_IMAGING_CEPH_ROOT", "/space/ceph/1/ABCD/users/arigby/2026_bmi_imaging_analysis"))
RESULTS_RUN = os.environ.get("BMI_IMAGING_RESULTS_RUN", "results_2026_06_29")
SHARED_DATAFILES = CEPH_ROOT / "shared" / "datafiles"
DATAFILE = Path(os.environ.get(
    "BMI_EXPLORE_DATAFILE",
    SHARED_DATAFILES / "datafile6.0_before_bfs_anthroqc_bmip_agediscrepancy_dmri_incl.txt",
))

PHENOS_TO_RUN = ["bmip", "perc_median"]

PHENO_META = {
    "bmip": {"analysis_dir": "bmip", "xname": "bmi"},
    "perc_median": {"analysis_dir": "perc_median", "xname": "perc_median"},
}

MODALITIES = ["rni"]
EFFECT_TYPE = "splines"
CORRECTION = "fdr"  # bonferroni | fdr | all_voxels
LATERALITY = ["non_lateralized", "bilateral"]

# CDC weight-class windows use minmax bmip -> perc_median mapping only
CDC_MAPPING_METHOD = "minmax"

CDC_BMIP_CLASS_DEFS = [
    ("underweight", "Underweight (<5th)", 0, 4),
    ("healthy", "Healthy (5th to <85th)", 5, 84),
    ("overweight", "Overweight (85th to <95th)", 85, 94),
    ("obesity", "Obesity (>=95th)", 95, 100),
]

COHORT_SEX_TO_DESIGNMAT = {"F": "female", "M": "male"}

print("RESULTS_RUN:", RESULTS_RUN)
print("DATAFILE exists:", DATAFILE.exists())


In [ ]:
# --- Path, mapping, discovery, and gradient helpers ---

def results_dir_for_pheno(pheno: str) -> Path:
    meta = PHENO_META[pheno]
    return CEPH_ROOT / meta["analysis_dir"] / "results" / RESULTS_RUN


def roi_data_dir_for_pheno(pheno: str) -> Path:
    return results_dir_for_pheno(pheno) / "roi_data_for_plots"


def export_dir_for_pheno(pheno: str) -> Path:
    out = results_dir_for_pheno(pheno) / "gradient_analysis"
    out.mkdir(parents=True, exist_ok=True)
    return out


def load_mapping_df(datafile: Path = DATAFILE) -> pd.DataFrame:
    need = ["participant_id", "session_id", "bmip", "perc_median", "ab_g_stc__cohort_sex"]
    header = pd.read_csv(datafile, sep="\t", nrows=0).columns
    missing = [c for c in need if c not in header]
    if missing:
        raise ValueError(f"Missing columns {missing} in {datafile}")
    df = pd.read_csv(datafile, sep="\t", usecols=need, low_memory=False)
    return df.dropna(subset=["bmip", "perc_median"]).rename(columns={"ab_g_stc__cohort_sex": "sex"})


def perc_median_range_in_bmip_band(sub: pd.DataFrame):
    """Min/max perc_median among visits in a bmip band (minmax CDC mapping)."""
    if sub.empty:
        raise ValueError("No visits in bmip band for mapping")
    pm = sub["perc_median"]
    return float(pm.min()), float(pm.max())


def build_cdc_perc_median_mapping(mapping_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for cohort_sex, designmat_sex in COHORT_SEX_TO_DESIGNMAT.items():
        sex_df = mapping_df[mapping_df["sex"] == cohort_sex]
        for cdc_class, cdc_label, bmip_lo, bmip_hi in CDC_BMIP_CLASS_DEFS:
            sub = sex_df[(sex_df["bmip"] >= bmip_lo) & (sex_df["bmip"] <= bmip_hi)]
            if sub.empty:
                pm_lo, pm_hi = np.nan, np.nan
                n_visits = 0
            else:
                pm_lo, pm_hi = perc_median_range_in_bmip_band(sub)
                n_visits = int(len(sub))
            rows.append({
                "cdc_class": cdc_class,
                "cdc_label": cdc_label,
                "sex": cohort_sex,
                "designmat_sex": designmat_sex,
                "bmip_range_low": bmip_lo,
                "bmip_range_high": bmip_hi,
                "perc_median_range_low": pm_lo,
                "perc_median_range_high": pm_hi,
                "mapping_method": CDC_MAPPING_METHOD,
                "n_visits": n_visits,
            })
    return pd.DataFrame(rows)


def cdc_segment_name(cdc_class: str) -> str:
    return f"cdc_{cdc_class}"


def build_cdc_bmip_segments() -> dict:
    return {cdc_segment_name(cls): (lo, hi) for cls, _label, lo, hi in CDC_BMIP_CLASS_DEFS}


def build_cdc_perc_median_segments_by_sex(cdc_mapping_table: pd.DataFrame) -> dict:
    by_sex = {}
    for designmat_sex in COHORT_SEX_TO_DESIGNMAT.values():
        segs = {}
        sub = cdc_mapping_table[cdc_mapping_table["designmat_sex"] == designmat_sex]
        for row in sub.itertuples(index=False):
            if pd.isna(row.perc_median_range_low) or pd.isna(row.perc_median_range_high):
                continue
            segs[cdc_segment_name(row.cdc_class)] = (
                float(row.perc_median_range_low),
                float(row.perc_median_range_high),
            )
        by_sex[designmat_sex] = segs
    return by_sex


def infer_sex_from_designmat(designmat_name: str) -> str:
    lower = designmat_name.lower()
    if "female" in lower:
        return "female"
    if "male" in lower:
        return "male"
    return "unknown"


def parse_roi_from_filename(file_name: str, modality: str, effect_type: str, xname: str) -> str:
    prefix = f"{modality}_{effect_type}_{xname}_derivatives_CI_"
    if file_name.startswith(prefix) and file_name.endswith(".txt"):
        return file_name[len(prefix):-4]
    return file_name


def find_derivative_files(base_dir: Path, modalities, effect_type, xname, correction, laterality_list):
    records = []
    if not base_dir.exists():
        return pd.DataFrame(records)
    for designmat_dir in sorted([d for d in base_dir.iterdir() if d.is_dir()]):
        sex = infer_sex_from_designmat(designmat_dir.name)
        for laterality in laterality_list:
            derivative_dir = designmat_dir / "derivatives" / laterality / correction
            if not derivative_dir.exists():
                continue
            pattern = f"*_{effect_type}_{xname}_derivatives_CI_*.txt"
            for fp in sorted(derivative_dir.glob(pattern)):
                for modality in modalities:
                    if not fp.name.startswith(f"{modality}_"):
                        continue
                    roi = parse_roi_from_filename(fp.name, modality, effect_type, xname)
                    records.append({
                        "file_path": fp,
                        "designmat": designmat_dir.name,
                        "sex": sex,
                        "laterality": laterality,
                        "modality": modality,
                        "roi": roi,
                    })
    return pd.DataFrame(records)


def subset_bin(df: pd.DataFrame, low: float, high: float) -> pd.DataFrame:
    return df[(df["xvec"] >= low) & (df["xvec"] <= high)].copy()


def summarize_gradient(df: pd.DataFrame, value_col="mean_derivative") -> dict:
    vals = pd.to_numeric(df[value_col], errors="coerce").dropna().to_numpy()
    if vals.size == 0:
        return {
            "n_points": 0,
            "mean_signed": np.nan,
            "mean_abs": np.nan,
            "median_abs": np.nan,
            "max_abs": np.nan,
        }
    abs_vals = np.abs(vals)
    return {
        "n_points": int(vals.size),
        "mean_signed": float(np.mean(vals)),
        "mean_abs": float(np.mean(abs_vals)),
        "median_abs": float(np.median(abs_vals)),
        "max_abs": float(np.max(abs_vals)),
    }


def cdc_segments_for_file(pheno: str, sex: str) -> dict:
    if pheno == "bmip":
        return dict(cdc_bmip_segments)
    if pheno == "perc_median":
        return dict(cdc_perc_segments_by_sex.get(sex, {}))
    return {}


def accumulate_cdc_wide(wide: dict, df: pd.DataFrame, segments: dict):
    for seg_name, (lo, hi) in segments.items():
        stats = summarize_gradient(subset_bin(df, lo, hi))
        for stat_key, stat_val in stats.items():
            wide[f"{seg_name}_{stat_key}"] = stat_val


In [ ]:
# --- Build CDC mapping, summarize gradients, export Table 2 CSVs ---

mapping_df = load_mapping_df()
print(f"Mapping cohort visits: {len(mapping_df):,}")

cdc_mapping_table = build_cdc_perc_median_mapping(mapping_df)
cdc_perc_segments_by_sex = build_cdc_perc_median_segments_by_sex(cdc_mapping_table)
cdc_bmip_segments = build_cdc_bmip_segments()

print(f"\nCDC bmip -> perc_median mapping ({CDC_MAPPING_METHOD}):")
print(cdc_mapping_table.to_string(index=False))

print("\nCDC bmip segments (both sexes):")
for name, (lo, hi) in cdc_bmip_segments.items():
    print(f"  {name}: [{lo}, {hi}]")

print(f"\nCDC perc_median segments by sex ({CDC_MAPPING_METHOD}):")
for sex, segs in cdc_perc_segments_by_sex.items():
    print(f"  {sex}:")
    for name, (lo, hi) in segs.items():
        print(f"    {name}: [{lo:.2f}, {hi:.2f}]")

# Mapping CSV once under perc_median (or first pheno if perc_median not run)
mapping_export_pheno = "perc_median" if "perc_median" in PHENOS_TO_RUN else PHENOS_TO_RUN[0]
mapping_out = export_dir_for_pheno(mapping_export_pheno)
# cdc_bmip_to_perc_median_mapping_minmax_fdr.csv
cdc_map_csv = mapping_out / f"cdc_bmip_to_perc_median_mapping_minmax_{CORRECTION}.csv"
cdc_mapping_table.to_csv(cdc_map_csv, index=False)
print("\nSaved mapping table:", cdc_map_csv)

# Wide CDC gradient summaries per pheno
all_summaries_cdc = []

for pheno in PHENOS_TO_RUN:
    meta = PHENO_META[pheno]
    xname = meta["xname"]
    base_dir = roi_data_dir_for_pheno(pheno)

    files_df = find_derivative_files(
        base_dir, MODALITIES, EFFECT_TYPE, xname, CORRECTION, LATERALITY
    )
    print(f"\n{pheno}: {len(files_df)} derivative files under {base_dir}")
    if files_df.empty:
        continue

    for row in files_df.itertuples(index=False):
        df = pd.read_csv(row.file_path)
        if not {"xvec", "mean_derivative"}.issubset(df.columns):
            continue

        cdc_segs = cdc_segments_for_file(pheno, row.sex)
        if not cdc_segs:
            continue

        wide_cdc = {
            "pheno": pheno,
            "xname": xname,
            "designmat": row.designmat,
            "sex": row.sex,
            "laterality": row.laterality,
            "modality": row.modality,
            "roi": row.roi,
            "file_path": str(row.file_path),
            "mapping_method": CDC_MAPPING_METHOD,
        }
        accumulate_cdc_wide(wide_cdc, df, cdc_segs)
        all_summaries_cdc.append(wide_cdc)

gradient_summary_cdc = pd.DataFrame(all_summaries_cdc)
print("\nCDC wide rows:", len(gradient_summary_cdc))

if gradient_summary_cdc.empty:
    print("Nothing to export.")
else:
    for pheno in PHENOS_TO_RUN:
        out_dir = export_dir_for_pheno(pheno)
        wide_cdc = gradient_summary_cdc[gradient_summary_cdc["pheno"] == pheno].copy()
        if wide_cdc.empty:
            continue
        tag = f"{pheno}_{CORRECTION}"
        # gradient_summary_wide_cdc_minmax_{bmip|perc_median}_fdr.csv
        out_csv = out_dir / f"gradient_summary_wide_cdc_minmax_{tag}.csv"
        wide_cdc.to_csv(out_csv, index=False)
        print(f"Saved CDC wide CSV ({len(wide_cdc)} rows):", out_csv)

    print("\nPreview:")
    display(gradient_summary_cdc.head(10))
